# two-optimizers-alternating-step — worked example 1: Alternating D-step then G-step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `two-optimizers-alternating-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A GAN trains two networks with two separate optimizers. Each iteration runs a discriminator step (zero_grad → D loss → D_opt.step) then a generator step (zero_grad → G loss → G_opt.step). The discriminator's fake input is detached so its gradient never flows into the generator during the D-step.

## Worked solution

We run one canonical GAN iteration on toy modules.

1. D-step: we call `D_opt.zero_grad()`, build `fake = G(z).detach()` so the generator is frozen here, compute a simplified Wasserstein loss `(D(fake) - D(x_real)).mean()`, backprop, and step `D_opt`.
2. G-step: we call `G_opt.zero_grad()`, recompute `fake = G(z)` with NO detach so gradients reach `G`, compute `-D(fake).mean()` (the generator wants D to score fakes high), backprop, and step `G_opt`.
3. Each optimizer only touches its own network's parameters, so the two updates are isolated.

We print both loss values to confirm the iteration ran end to end.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)
G = nn.Linear(4, 6)
D = nn.Linear(6, 1)
G_opt = t.optim.SGD(G.parameters(), lr=0.01)
D_opt = t.optim.SGD(D.parameters(), lr=0.01)

def gan_iter(G, D, G_opt, D_opt, z, x_real):
    D_opt.zero_grad()
    fake = G(z).detach()
    loss_D = (D(fake) - D(x_real)).mean()
    loss_D.backward()
    D_opt.step()

    G_opt.zero_grad()
    fake = G(z)
    loss_G = -D(fake).mean()
    loss_G.backward()
    G_opt.step()
    return loss_D.item(), loss_G.item()

z = t.randn(8, 4)
x_real = t.randn(8, 6)
lD, lG = gan_iter(G, D, G_opt, D_opt, z, x_real)
print('loss_D:', round(lD, 4))
print('loss_G:', round(lG, 4))